# Feature engineering del Modelo A limpio

Este notebook construye las variables de entrada para el Modelo A del sensor virtual.

El objetivo es crear un dataset de features a partir de `dataset_modelable_10s.csv`, manteniendo una regla metodológica clave:

- no usar `LV411`;
- no usar `SP_LT411`;
- no usar `LT411` retrasado;
- no usar variables derivadas de `LT411` como entrada del modelo.

De esta forma se evita que el modelo copie directamente el lazo de control o el propio sensor que se quiere auditar.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Cargamos el dataset modelable a 10 segundos

df = pd.read_csv("data_limpio/dataset_modelable_10s.csv")

df["Time"] = pd.to_datetime(df["Time"])

df.head()

,Time,LT411,SP_LT411,LV411,DT412,INT_P101,FQC400_1,SP_VAPOR,FV400_1,PT442,SP_PT442,PV442,TT413,TT415,LT426,SP_LT426,ACCION_PCT02,FT428,PIT410,PIT414
0,2021-11-05 08:00:00,58.142360,60.0,100.0,1229.492151,58.408563,3054.785938,69.0,59.5,0.154167,0.14,0.0,80.976565,81.82292,50.439814,50.0,18.861438,6.266928,0.326085,0.579514
1,2021-11-05 08:00:10,57.962962,60.0,100.0,1229.049451,58.272569,3052.526001,69.0,59.5,0.154282,0.14,0.0,80.954863,81.81424,50.422452,50.0,19.102801,6.293403,0.326181,0.579456
2,2021-11-05 08:00:20,57.323494,60.0,100.0,1229.088501,58.234953,3038.494727,69.0,59.5,0.154225,0.14,0.0,80.946182,81.81424,50.434027,50.0,19.315793,6.325521,0.326403,0.579051
3,2021-11-05 08:00:30,57.335069,60.0,100.0,1228.762976,58.463541,3031.794189,69.0,59.5,0.154282,0.14,0.0,80.946182,81.81424,50.422452,50.0,19.470506,6.357205,0.326340,0.578588
4,2021-11-05 08:00:40,57.835647,60.0,100.0,1229.453088,58.159721,3036.406226,69.0,59.5,0.154485,0.14,0.0,80.946182,81.81424,50.428239,50.0,19.712129,6.391927,0.326403,0.578125


In [3]:
# Ordenamos el dataset por tiempo

df = df.sort_values("Time").reset_index(drop=True)

## Split temporal

Se mantiene el split temporal definido en el EDA:

- Día 1: train
- Día 2: validación
- Día 3: test

No se utiliza split aleatorio porque el problema es temporal. El Día 3 se reserva para evaluar comportamiento anómalo y no se usará para entrenar el Modelo A.

In [4]:
# Definimos cortes temporales para train, validación y test

inicio_train = pd.Timestamp("2021-11-05 08:00:00")
fin_train = pd.Timestamp("2021-11-06 07:59:59")

inicio_valid = pd.Timestamp("2021-11-06 08:00:00")
fin_valid = pd.Timestamp("2021-11-07 07:59:59")

inicio_test = pd.Timestamp("2021-11-07 08:00:00")
fin_test = df["Time"].max()

In [5]:
# Asignamos cada fila a train, validación o test

df["bloque"] = "sin_asignar"

df.loc[(df["Time"] >= inicio_train) & (df["Time"] <= fin_train), "bloque"] = "train"
df.loc[(df["Time"] >= inicio_valid) & (df["Time"] <= fin_valid), "bloque"] = "validacion"
df.loc[(df["Time"] >= inicio_test) & (df["Time"] <= fin_test), "bloque"] = "test"

df["bloque"].value_counts()

bloque
train         8640
validacion    8640
test          5790
Name: count, dtype: int64

In [6]:
# Revisamos inicio, fin y filas por bloque

df.groupby("bloque").agg(
    inicio=("Time", "min"),
    fin=("Time", "max"),
    filas=("Time", "count")
)

,inicio,fin,filas
bloque,,,
test,2021-11-07 08:00:00,2021-11-08 00:04:50,5790
train,2021-11-05 08:00:00,2021-11-06 07:59:50,8640
validacion,2021-11-06 08:00:00,2021-11-07 07:59:50,8640


## Corrección física de FQC400_1

En limpieza se conservó la señal original para mantener trazabilidad.

Aquí se crean dos variables nuevas:

- `FQC400_1_corr`: caudal corregido, llevando valores negativos a 0.
- `FQC400_1_negativo_flag`: indicador de valores negativos en la señal original.

No se eliminan filas.

In [7]:
# Creamos corrección física y flag para FQC400_1

df["FQC400_1_corr"] = df["FQC400_1"].clip(lower=0)

df["FQC400_1_negativo_flag"] = (df["FQC400_1"] < 0).astype(int)

print("Valores negativos originales:", df["FQC400_1_negativo_flag"].sum())
print("Mínimo FQC400_1 original:", df["FQC400_1"].min())
print("Mínimo FQC400_1_corr:", df["FQC400_1_corr"].min())

Valores negativos originales: 61
Mínimo FQC400_1 original: -0.8061619043350214
Mínimo FQC400_1_corr: 0.0


## Variables físicas derivadas

Se crean variables derivadas con sentido físico:

- `DELTA_TT = TT413 - TT415`
- `DELTA_PIT = PIT414 - PIT410`

Estas variables ayudan a representar diferencias térmicas y de presión del proceso sin usar la señal objetivo `LT411` como entrada.

In [8]:
# Creamos variables físicas derivadas

df["DELTA_TT"] = df["TT413"] - df["TT415"]

df["DELTA_PIT"] = df["PIT414"] - df["PIT410"]

df[["DELTA_TT", "DELTA_PIT"]].head()

,DELTA_TT,DELTA_PIT
0,-0.846355,0.253429
1,-0.859377,0.253275
2,-0.868057,0.252648
3,-0.868057,0.252248
4,-0.868057,0.251722


## Revisión de setpoints

Los setpoints se revisan para confirmar si son constantes.

Si son constantes, no aportan información útil al modelo y se excluyen del Modelo A.

In [9]:
# Revisamos si los setpoints son constantes

setpoints = ["SP_LT411", "SP_VAPOR", "SP_PT442", "SP_LT426"]

for columna in setpoints:
    print(columna, "valores únicos:", df[columna].nunique())

SP_LT411 valores únicos: 1
SP_VAPOR valores únicos: 1
SP_PT442 valores únicos: 1
SP_LT426 valores únicos: 1


## Definición del Modelo A limpio

El Modelo A se construye sin variables con riesgo claro de leakage:

- `LV411`: válvula asociada al lazo de control de LT411.
- `SP_LT411`: setpoint del lazo de nivel.
- `LT411_lagged`: no se usan retardos de la propia variable objetivo.
- `delta_LT411`: no se usa ninguna derivada de la variable objetivo como feature.

La variable `LT411` se conserva únicamente como target.

In [10]:
# Definimos la variable objetivo

target = "LT411"

In [11]:
# Definimos variables excluidas del Modelo A por riesgo de leakage o circularidad

variables_excluidas_modelo_A = [
    "LT411",
    "LV411",
    "SP_LT411"
]

In [13]:
# Definimos variables para el Modelo A limpio

variables_modelo_A = [
    "DT412",
    "INT_P101",
    "FQC400_1_corr",
    "FQC400_1_negativo_flag",
    "FV400_1",
    "PT442",
    "PV442",
    "TT413",
    "TT415",
    "LT426",
    "ACCION_PCT02",
    "FT428",
    "PIT410",
    "PIT414",
    "DELTA_TT",
    "DELTA_PIT"
]

variables_modelo_A

['DT412',
 'INT_P101',
 'FQC400_1_corr',
 'FQC400_1_negativo_flag',
 'FV400_1',
 'PT442',
 'PV442',
 'TT413',
 'TT415',
 'LT426',
 'ACCION_PCT02',
 'FT428',
 'PIT410',
 'PIT414',
 'DELTA_TT',
 'DELTA_PIT']

In [15]:
# Creamos el dataset base para feature engineering

columnas_base = ["Time", "bloque", target] + variables_modelo_A

df_base = df[columnas_base].copy()

df_base.head()

,Time,bloque,LT411,DT412,INT_P101,FQC400_1_corr,FQC400_1_negativo_flag,FV400_1,PT442,PV442,TT413,TT415,LT426,ACCION_PCT02,FT428,PIT410,PIT414,DELTA_TT,DELTA_PIT
0,2021-11-05 08:00:00,train,58.142360,1229.492151,58.408563,3054.785938,0,59.5,0.154167,0.0,80.976565,81.82292,50.439814,18.861438,6.266928,0.326085,0.579514,-0.846355,0.253429
1,2021-11-05 08:00:10,train,57.962962,1229.049451,58.272569,3052.526001,0,59.5,0.154282,0.0,80.954863,81.81424,50.422452,19.102801,6.293403,0.326181,0.579456,-0.859377,0.253275
2,2021-11-05 08:00:20,train,57.323494,1229.088501,58.234953,3038.494727,0,59.5,0.154225,0.0,80.946182,81.81424,50.434027,19.315793,6.325521,0.326403,0.579051,-0.868057,0.252648
3,2021-11-05 08:00:30,train,57.335069,1228.762976,58.463541,3031.794189,0,59.5,0.154282,0.0,80.946182,81.81424,50.422452,19.470506,6.357205,0.326340,0.578588,-0.868057,0.252248
4,2021-11-05 08:00:40,train,57.835647,1229.453088,58.159721,3036.406226,0,59.5,0.154485,0.0,80.946182,81.81424,50.428239,19.712129,6.391927,0.326403,0.578125,-0.868057,0.251722


## Lags, rolling y deltas temporales

Se amplía la memoria temporal del modelo respecto a la primera versión.

Como el dataset está a 10 segundos:

- lags `[1, 2, 3, 6, 12, 18]` equivalen a 10s, 20s, 30s, 1min, 2min y 3min.
- rolling `[6, 30, 90]` equivalen a 1min, 5min y 15min.

Las variables rolling se calculan siempre hacia atrás, sin mirar al futuro.

In [16]:
# Definimos lags y ventanas rolling para el primer Modelo A


lags = [1, 2, 3, 6, 12, 18]

ventanas_rolling = [6, 30, 90]

In [17]:
# Definimos variables a las que aplicaremos lags

variables_lag = variables_modelo_A.copy()

variables_lag

['DT412',
 'INT_P101',
 'FQC400_1_corr',
 'FQC400_1_negativo_flag',
 'FV400_1',
 'PT442',
 'PV442',
 'TT413',
 'TT415',
 'LT426',
 'ACCION_PCT02',
 'FT428',
 'PIT410',
 'PIT414',
 'DELTA_TT',
 'DELTA_PIT']

In [18]:
# Definimos variables principales para rolling features

variables_rolling = [
    "DT412",
    "INT_P101",
    "FQC400_1_corr",
    "PT442",
    "TT413",
    "TT415",
    "PIT410",
    "PIT414",
    "DELTA_TT",
    "DELTA_PIT"
]

variables_rolling

['DT412',
 'INT_P101',
 'FQC400_1_corr',
 'PT442',
 'TT413',
 'TT415',
 'PIT410',
 'PIT414',
 'DELTA_TT',
 'DELTA_PIT']

In [19]:
# Definimos variables para deltas temporales

variables_delta = [
    "DT412",
    "INT_P101",
    "FQC400_1_corr",
    "PT442",
    "TT413",
    "TT415",
    "PIT410",
    "PIT414",
    "DELTA_TT",
    "DELTA_PIT"
]

variables_delta

['DT412',
 'INT_P101',
 'FQC400_1_corr',
 'PT442',
 'TT413',
 'TT415',
 'PIT410',
 'PIT414',
 'DELTA_TT',
 'DELTA_PIT']

In [20]:
# Creamos una función simple para generar lags, rolling y deltas por bloque

def crear_features_temporales(datos):
    datos = datos.sort_values("Time").copy()

    for variable in variables_lag:
        for lag in lags:
            datos[variable + "_lag_" + str(lag)] = datos[variable].shift(lag)

    for variable in variables_rolling:
        for ventana in ventanas_rolling:
            datos[variable + "_roll_mean_" + str(ventana)] = datos[variable].rolling(window=ventana).mean()
            datos[variable + "_roll_std_" + str(ventana)] = datos[variable].rolling(window=ventana).std()
            datos[variable + "_roll_range_" + str(ventana)] = (
                datos[variable].rolling(window=ventana).max() - 
                datos[variable].rolling(window=ventana).min()
            )

    for variable in variables_delta:
        datos[variable + "_delta_1"] = datos[variable].diff(1)
        datos[variable + "_delta_6"] = datos[variable] - datos[variable].shift(6)

    return datos

## Creación de features por bloque

Las features temporales se calculan por separado en train, validación y test.

Esto evita que los lags o rolling de un bloque tomen información del bloque anterior.

In [22]:
# Creamos features temporales separando train, validación y test

df_train_feat = crear_features_temporales(df_base[df_base["bloque"] == "train"])

df_valid_feat = crear_features_temporales(df_base[df_base["bloque"] == "validacion"])

df_test_feat = crear_features_temporales(df_base[df_base["bloque"] == "test"])

C:\Users\Dario\AppData\Local\Temp\ipykernel_20160\546102436.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  datos[variable + "_roll_std_" + str(ventana)] = datos[variable].rolling(window=ventana).std()
C:\Users\Dario\AppData\Local\Temp\ipykernel_20160\546102436.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  datos[variable + "_roll_range_" + str(ventana)] = (
C:\Users\Dario\AppData\Local\Temp\ipykernel_20160\546102436.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

In [23]:
# Unimos los bloques con features temporales

df_features = pd.concat([df_train_feat, df_valid_feat, df_test_feat], axis=0)

df_features = df_features.sort_values("Time").reset_index(drop=True)

df_features.head()

,Time,bloque,LT411,DT412,INT_P101,FQC400_1_corr,FQC400_1_negativo_flag,FV400_1,PT442,PV442,...,TT415_delta_1,TT415_delta_6,PIT410_delta_1,PIT410_delta_6,PIT414_delta_1,PIT414_delta_6,DELTA_TT_delta_1,DELTA_TT_delta_6,DELTA_PIT_delta_1,DELTA_PIT_delta_6
0,2021-11-05 08:00:00,train,58.142360,1229.492151,58.408563,3054.785938,0,59.5,0.154167,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-11-05 08:00:10,train,57.962962,1229.049451,58.272569,3052.526001,0,59.5,0.154282,0.0,...,-0.008681,NaN,0.000095,NaN,-0.000058,NaN,-0.013021,NaN,-0.000153,NaN
2,2021-11-05 08:00:20,train,57.323494,1229.088501,58.234953,3038.494727,0,59.5,0.154225,0.0,...,0.000000,NaN,0.000223,NaN,-0.000405,NaN,-0.008681,NaN,-0.000628,NaN
3,2021-11-05 08:00:30,train,57.335069,1228.762976,58.463541,3031.794189,0,59.5,0.154282,0.0,...,0.000000,NaN,-0.000064,NaN,-0.000463,NaN,0.000000,NaN,-0.000399,NaN
4,2021-11-05 08:00:40,train,57.835647,1229.453088,58.159721,3036.406226,0,59.5,0.154485,0.0,...,0.000000,NaN,0.000064,NaN,-0.000463,NaN,0.000000,NaN,-0.000527,NaN


## Auditoría de nulos generados por lags y rolling

Los lags, rolling y deltas generan nulos al inicio de cada bloque.

Esto es esperado. Se eliminan esas filas con `dropna()` para dejar una matriz de entrenamiento consistente.

In [24]:
# Revisamos tamaño antes de eliminar nulos generados por lags y rolling

filas_antes_dropna = df_features.shape[0]

print("Filas antes de dropna:", filas_antes_dropna)
print("Columnas:", df_features.shape[1])
print("Nulos totales:", df_features.isnull().sum().sum())

Filas antes de dropna: 23070
Columnas: 225
Nulos totales: 13296


In [25]:
# Eliminamos filas con nulos generados por lags y rolling

df_features_limpio = df_features.dropna().copy()

In [26]:
# Revisamos filas por bloque después de dropna

df_features_limpio.groupby("bloque").agg(
    inicio=("Time", "min"),
    fin=("Time", "max"),
    filas=("Time", "count")
)

,inicio,fin,filas
bloque,,,
test,2021-11-07 08:14:50,2021-11-08 00:04:50,5701
train,2021-11-05 08:14:50,2021-11-06 07:59:50,8551
validacion,2021-11-06 08:14:50,2021-11-07 07:59:50,8551


In [27]:
# Guardamos el dataset preparado para el Modelo A

df_features_limpio.to_csv("data_limpio/features_modelo_A.csv", index=False)

## Conclusiones de feature engineering

Se ha creado una segunda versión de features para el Modelo A limpio.

Las principales mejoras respecto a la primera versión son:

- corrección física de `FQC400_1` mediante `FQC400_1_corr`;
- creación de `FQC400_1_negativo_flag`;
- incorporación de `DELTA_TT` y `DELTA_PIT`;
- ampliación de lags hasta 3 minutos;
- incorporación de rolling de 1, 5 y 15 minutos;
- creación de rolling mean, rolling std y rolling range;
- creación de deltas temporales de variables físicas.

El dataset final se guarda como:

- `data_limpio/features_modelo_A.csv`

El siguiente paso será reentrenar y comparar modelos usando esta versión de features.